In [1]:
import subprocess, sys
for pkg in ['pyspark==3.5.1', 'sentence-transformers', 'anthropic', 'pyarrow']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])



In [ ]:
import os, json, warnings, random
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.cluster import MiniBatchKMeans
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType

warnings.filterwarnings('ignore')

CFG = dict(
 
    input_parquet         = '/content/drive/MyDrive/Office_Products_Cleaned_v2.parquet',
    output_dir            = '/content/drive/MyDrive',
    report_dir            = '/content/drive/MyDrive',


    category_name         = 'Office_Products',
    n_sample              = 1_000_000,  
    num_clusters          = 100,
    sentences_per_cluster = 8,          
    embed_batch_size      = 512,
    embed_model           = 'sentence-transformers/all-MiniLM-L6-v2',
    random_seed           = 42,
    spark_driver_memory   = '12g',
)

Path(CFG['output_dir']).mkdir(parents=True, exist_ok=True)
Path(CFG['report_dir']).mkdir(parents=True, exist_ok=True)


In [3]:
import os
import shutil
from google.colab import drive

mountpoint = '/content/drive'


os.makedirs(mountpoint, exist_ok=True)

if os.path.ismount(mountpoint):
    print(f"'{mountpoint}' is already mounted. Attempting to force remount.")
else:
    if os.path.exists(mountpoint) and os.listdir(mountpoint):
        print(f"'{mountpoint}' is not mounted but contains files. Clearing its contents...")
        for item in os.listdir(mountpoint):
            item_path = os.path.join(mountpoint, item)
            try:
                if os.path.isfile(item_path) or os.path.islink(item_path):
                    os.unlink(item_path)
                elif os.path.isdir(item_path):

                    shutil.rmtree(item_path)
            except Exception as e:
                print(f"Error removing {item_path}: {e}")
    else:
        print(f"'{mountpoint}' is not mounted and is empty or does not exist.")

drive.mount(mountpoint, force_remount=True)
print('✓ Google Drive mounted')

'/content/drive' is not mounted but contains files. Clearing its contents...
Mounted at /content/drive
✓ Google Drive mounted


In [4]:
spark = (
    SparkSession.builder
    .appName('Phase3_OfficeProducts_HardLabeling')
    .master('local[*]')
    .config('spark.driver.memory',          CFG['spark_driver_memory'])
    .config('spark.driver.maxResultSize',   '8g')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.sql.adaptive.enabled',   'true')
    .config('spark.serializer', 'org.apache.spark.serializer.KryoSerializer')
    .config('spark.sql.parquet.compression.codec', 'snappy')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

df_full = spark.read.parquet(CFG['input_parquet'])
total_rows = df_full.count()
print(f'{total_rows:,} rows')
df_full.printSchema()
df_full.show(3, truncate=80)


13,915,549 rows
root
 |-- parent_asin: string (nullable = true)
 |-- review_id: long (nullable = true)
 |-- sentence_id: integer (nullable = true)
 |-- sentence_text: string (nullable = true)
 |-- rating: double (nullable = true)

+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
|parent_asin|review_id|sentence_id|                                                                   sentence_text|rating|
+-----------+---------+-----------+--------------------------------------------------------------------------------+------+
| B07N4C799S|        4|          1|                        rings were a [GENERIC_NOUN] big but [GENERIC_NOUN] crazy|   5.0|
| B07RGP45LB|        5|          1|comes with i think 3 different interchangeable color bands which i wasnt expe...|   5.0|
| B083H1KV7Z|       19|          1|let me start by saying the quality of these cards is amazing the design is be...|   5.0|
+-----------+---------+--

In [5]:
# Sample ~1M dòng ngẫu nhiên (nếu data < 1M thì lấy hết)
frac = min(CFG['n_sample'] / max(total_rows, 1), 1.0)
df_sampled = df_full.sample(fraction=frac, seed=CFG['random_seed'])
n_sampled  = df_sampled.count()
print(f' Sample: {total_rows:,} → {n_sampled:,} câu (frac={frac:.4f})')

# Collect về Pandas, bỏ review_id theo yêu cầu output
df_pd = (
    df_sampled
    .select('parent_asin','review_id', 'sentence_id', 'sentence_text', 'rating')
    .toPandas()
)
df_pd.head(5)


 Sample: 13,915,549 → 999,618 câu (frac=0.0719)


,parent_asin,review_id,sentence_id,sentence_text,rating
0,B00PT7YRBM,28,1,asinb00pt7yrbm the ultimate preschool curricul...,5.0
1,B00G02DIYC,34,3,its a pretty [GENERIC_NOUN] to tell people to ...,5.0
2,B01GPWV6PE,161,1,it is my number 1 preferred notelogging book f...,5.0
3,B0BXCBR4KJ,183,1,17 per mailer works as [GENERIC_NOUN] would ex...,5.0
4,B0C6765BDD,201,2,the other [GENERIC_NOUN] works but is very dry,1.0


## Bước 3: Sinh Embedding bằng SentenceTransformer

In [6]:
print(f'Loading {CFG["embed_model"]} ...')
embedder = SentenceTransformer(CFG['embed_model'])

print(f'Encoding {len(df_pd):,} câu (batch={CFG["embed_batch_size"]}) ...')
embeddings = embedder.encode(
    df_pd['sentence_text'].tolist(),
    batch_size           = CFG['embed_batch_size'],
    show_progress_bar    = True,
    normalize_embeddings = True,   # tốt hơn cho cosine distance
    convert_to_numpy     = True,
)
print(f'Embeddings shape: {embeddings.shape}')


Loading sentence-transformers/all-MiniLM-L6-v2 ...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Encoding 999,618 câu (batch=512) ...


Batches:   0%|          | 0/1953 [00:00<?, ?it/s]

Embeddings shape: (999618, 384)


## Bước 4: Phân cụm 100 clusters bằng MiniBatchKMeans

In [7]:
print(f'Clustering {len(embeddings):,} embeddings → {CFG["num_clusters"]} clusters')

km = MiniBatchKMeans(
    n_clusters   = CFG['num_clusters'],
    random_state = CFG['random_seed'],
    batch_size   = 4096,
    max_iter     = 100,
    n_init       = 3,
    verbose      = 0,
)
df_pd['cluster_id'] = km.fit_predict(embeddings)

cc = df_pd['cluster_id'].value_counts()
print(f'  Min câu/cluster : {cc.min()}')
print(f'  Max câu/cluster : {cc.max()}')
print(f'  Mean câu/cluster: {cc.mean():.1f}')


Clustering 999,618 embeddings → 100 clusters
  Min câu/cluster : 1399
  Max câu/cluster : 28878
  Mean câu/cluster: 9996.2


## Bước 5: Chọn 8 câu ngẫu nhiên từ mỗi cluster

In [8]:
df_annotation = (
    df_pd
    .groupby('cluster_id', group_keys=False)
    .apply(lambda g: g.sample(
        n            = min(CFG['sentences_per_cluster'], len(g)),
        random_state = CFG['random_seed'],
    ))
    .reset_index(drop=True)
)

print(f' {len(df_annotation):,} câu được chọn để gán nhãn')
print(f'  ({CFG["num_clusters"]} clusters × {CFG["sentences_per_cluster"]} câu)')
df_annotation[['parent_asin','sentence_id','sentence_text','rating','cluster_id']].head(10)


 800 câu được chọn để gán nhãn
  (100 clusters × 8 câu)


,parent_asin,sentence_id,sentence_text,rating,cluster_id
0,B005DRVBIA,1,i will now start looking for [GENERIC_NOUN] wh...,1.0,0
1,B006SBBNV0,4,the phone worked great for the first 90 [GENER...,1.0,0
2,B07T6PGM4J,1,sending a pic from my phone does not allow me ...,3.0,0
3,B095CPWNTQ,4,previous voip providers ive had were vonage ti...,5.0,0
4,B0BMGB65D6,2,i wish i would have had this many [GENERIC_NOU...,5.0,0
5,B08D1BWLL7,1,great buy all the [GENERIC_NOUN] around love t...,5.0,0
6,B000RQD37E,3,only had it two [GENERIC_NOUN] and already the...,3.0,0
7,B08PY9F52L,1,battery life seemed to be solid i could go thr...,1.0,0
8,B08QCS9XPH,2,on the first use the blue marker broke apart a...,2.0,1
9,B0BN6DFMKQ,2,these markers glide and love the fine tip,4.0,1


In [17]:
# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

from openai import OpenAI

_api_key = os.environ.get("OPENAI_API_KEY", OPENAI_API_KEY)

client = OpenAI(api_key=_api_key)

In [18]:
import time # Add this line

SYSTEM_PROMPT = """You are an ASTE (Aspect-Sentiment-Opinion Triplet Extraction) expert for e-book and Kindle product reviews.

Extract all aspect-opinion-sentiment triplets from the given sentence.

## FORMAT
Return a JSON array of lists: [[aspect, opinion, sentiment], ...]
- aspect: the aspect term (noun/noun phrase being evaluated)
- opinion: the opinion expression (include negation if present, e.g. "not good", "not bad")
- sentiment: 0 (negative), 1 (neutral), 2 (positive)

## RULES
- If no aspect-opinion pair found, return: []
- Keep negation as part of the opinion (e.g. "not good" → sentiment 0, "not bad" → sentiment 2)
- If multiple aspects share the same opinion, create one triplet per aspect
- If one aspect has multiple opinions, create one triplet per opinion
- Implicit opinions (verbs/phrases) are valid: "I love X" → [["X", "love", 2]]
- Sentences with no specific aspect (e.g. "Very good", "I love it", "Absolutely terrible") → []
- Return ONLY the JSON array, no explanation, no markdown

## EXAMPLES
The story is engaging. → [["story", "engaging", 2]]
The ending is predictable. → [["ending", "predictable", 0]]
The story is okay. → [["story", "okay", 1]]
The screen is not good. → [["screen", "not good", 0]]
The keyboard is not bad. → [["keyboard", "not bad", 2]]
The app is not too slow. → [["app", "not too slow", 1]]
I love the keyboard. → [["keyboard", "love", 2]]
I hate the touchpad. → [["touchpad", "hate", 0]]
The app crashes often. → [["app", "crashes often", 0]]
The story is engaging but the ending is disappointing. → [["story", "engaging", 2], ["ending", "disappointing", 0]]
The sound and bass are amazing. → [["sound", "amazing", 2], ["bass", "amazing", 2]]
The plot and characters are boring. → [["plot", "boring", 0], ["characters", "boring", 0]]
The battery is long-lasting and reliable. → [["battery", "long-lasting", 2], ["battery", "reliable", 2]]
The plot is engaging, the writing is smooth, but the ending feels rushed. → [["plot", "engaging", 2], ["writing", "smooth", 2], ["ending", "rushed", 0]]
This book is better than my old one. → [["book", "better", 2]]
Very good. → []
I love it. → []
Absolutely terrible. → []"""

def extract_triplets(sentence: str, max_retries: int = 3) -> list:
    """Gọi GPT-4o-mini để trích xuất ASTE triplets. Trả về list hoặc [] nếu lỗi."""
    for attempt in range(max_retries):
        try:
            sentence = sentence.replace("[GENERIC_NOUN]", "").replace("[DOMAIN_NOISE]", "").replace("[NEG]", "").strip()
            response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": sentence},
                ],
                max_tokens=512,
                temperature=0,
            )
            raw = response.choices[0].message.content.strip()
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
            return json.loads(raw)
        except json.JSONDecodeError:
            if attempt == max_retries - 1:
                return []
        except Exception as e:
            err = str(e)
            if "rate_limit" in err.lower() or "429" in err:
                wait = 2 ** (attempt + 2)
                print(f"  Rate limit, waiting {wait}s...")
                time.sleep(wait)
            elif attempt == max_retries - 1:
                print(f"  Error on '{sentence[:40]}...': {e}")
                return []
            else:
                time.sleep(2 ** attempt)
    return []


triplets_all = []
t_start = time.time()

for idx, row in df_annotation.iterrows():
    triplets = extract_triplets(row["sentence_text"])
    triplets_all.append(triplets)

    done = len(triplets_all)
    if done % 100 == 0:
        elapsed = time.time() - t_start
        rate = done / elapsed
        remaining = (len(df_annotation) - done) / rate if rate > 0 else 0
        print(f"  [{done}/{len(df_annotation)}]  {rate:.1f} calls/s  ~{remaining:.0f}s remaining")

df_annotation = df_annotation.copy()
df_annotation["triplets"] = triplets_all
df_annotation["category_name"] = CFG['category_name']

n_with = sum(1 for t in triplets_all if t)
print(f"\nLabeling done in {time.time() - t_start:.1f}s  —  {n_with}/{len(triplets_all)} sentences with triplets")

  [100/800]  1.6 calls/s  ~441s remaining
  [200/800]  1.6 calls/s  ~374s remaining
  [300/800]  1.6 calls/s  ~314s remaining
  [400/800]  1.5 calls/s  ~263s remaining
  [500/800]  1.5 calls/s  ~202s remaining
  [600/800]  1.5 calls/s  ~136s remaining
  [700/800]  1.5 calls/s  ~68s remaining
  [800/800]  1.5 calls/s  ~0s remaining

Labeling done in 550.5s  —  495/800 sentences with triplets


In [24]:
df_annotation.head(5)

,parent_asin,review_id,sentence_id,sentence_text,rating,cluster_id,triplets,category_name
0,B005DRVBIA,42949972576,1,i will now start looking for [GENERIC_NOUN] wh...,1.0,0,[],Office_Products
1,B006SBBNV0,128849179742,4,the phone worked great for the first 90 [GENER...,1.0,0,"[[phone, worked great, 2]]",Office_Products
2,B07T6PGM4J,240518179598,1,sending a pic from my phone does not allow me ...,3.0,0,"[[pic, not allow me to edit the size, 0], [col...",Office_Products
3,B095CPWNTQ,197568501311,4,previous voip providers ive had were vonage ti...,5.0,0,"[[voip providers, tired of the price, 0], [oom...",Office_Products
4,B0BMGB65D6,94489289778,2,i wish i would have had this many [GENERIC_NOU...,5.0,0,[],Office_Products


In [36]:
# Thêm category_name, bỏ cluster_id
# df_annotation đã có category_name và triplets

# Tạo một bản sao để xử lý, tránh ảnh hưởng đến df_annotation gốc
df_out_pandas = df_annotation.copy()

# Chuyển đổi cột 'triplets' từ list of lists sang list of dictionaries

# Chuyển đổi Pandas DataFrame sang Spark DataFrame
df_spark_out = spark.createDataFrame(df_out_pandas)


out_path = str(Path(CFG['output_dir']) / f'labeled_data_{CFG["category_name"]}.parquet')
df_spark_out.write.mode('overwrite').parquet(out_path)

df_spark_out.printSchema()
df_spark_out.show(3, truncate=80)

root
 |-- parent_asin: string (nullable = true)
 |-- review_id: long (nullable = true)
 |-- sentence_id: long (nullable = true)
 |-- sentence_text: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- cluster_id: long (nullable = true)
 |-- triplets: array (nullable = true)
 |    |-- element: array (containsNull = true)
 |    |    |-- element: string (containsNull = true)
 |-- category_name: string (nullable = true)

+-----------+------------+-----------+--------------------------------------------------------------------------------+------+----------+--------------------------------------------------------------------------------+---------------+
|parent_asin|   review_id|sentence_id|                                                                   sentence_text|rating|cluster_id|                                                                        triplets|  category_name|
+-----------+------------+-----------+--------------------------------------------------------

In [ ]:
import pyspark.sql.functions as F

# Tính thống kê
# Chuyển đổi Spark DataFrame sang Pandas DataFrame để dễ dàng xử lý với logic hiện có
df_pd_stats = df_spark_out.toPandas()

# Cột 'triplets' trong df_pd_stats hiện có dạng list of lists, ví dụ: [['aspect', 'opinion', '2']]
# Chuyển đổi nó sang list of dictionaries, ví dụ: [{'aspect': '...', 'opinion': '...', 'sentiment': 2}]
# để các phép thống kê tiếp theo hoạt động chính xác.
def transform_triplets_for_stats(triplets_list_of_lists):
    processed_triplets = []
    for t_list in triplets_list_of_lists:
        if isinstance(t_list, list) and len(t_list) == 3:
            try:
                # Đảm bảo sentiment được chuyển đổi thành số nguyên
                processed_triplets.append({
                    'aspect': t_list[0],
                    'opinion': t_list[1],
                    'sentiment': int(t_list[2])
                })
            except (ValueError, IndexError):
                # Bỏ qua các triplet không hợp lệ hoặc có định dạng sai
                continue
    return processed_triplets

df_pd_stats['processed_triplets'] = df_pd_stats['triplets'].apply(transform_triplets_for_stats)

total     = len(df_pd_stats)
with_t    = df_pd_stats['processed_triplets'].apply(bool).sum()
all_t     = [t for ts in df_pd_stats['processed_triplets'] for t in ts]
n_t       = len(all_t)
multi_t   = df_pd_stats['processed_triplets'].apply(lambda x: len(x) > 1).sum()
max_t     = df_pd_stats['processed_triplets'].apply(len).max() if total else 0
avg_t     = n_t / __builtins__.max(with_t, 1)
sent_dist = {0:0, 1:0, 2:0}
for t in all_t:
    s = t.get('sentiment', -1)
    if s in sent_dist: sent_dist[s] += 1

total_t = __builtins__.sum(sent_dist.values())
print(f'Sentences annotated        : {total:,}')
print(f'Sentences với triplet      : {with_t:,}')
print(f'Sentences không có triplet : {total-with_t:,}')
print(f'Coverage                   : {100*with_t/__builtins__.max(total,1):.1f}%')
print(f'Total triplets             : {n_t:,}')
print(f'Avg triplets/sentence(w/t) : {avg_t:.2f}')
print(f'Sentences có >1 triplet    : {multi_t:,}')
print(f'Max triplets/sentence      : {max_t}')
print(f'Sentiment – Neg/Neu/Pos    : {sent_dist[0]:,} / {sent_dist[1]:,} / {sent_dist[2]:,}')


report_path = str(Path(CFG['report_dir']) / f'hard_labeling_report_{CFG["category_name"]}.txt')
with open(report_path, 'w', encoding='utf-8') as f:
    f.write('='*80 + '\n')
    f.write(f'HARD LABELING REPORT – {CFG["category_name"]}\n')
    f.write('='*80 + '\n')
    f.write(f'Generated: {datetime.now():%Y-%m-%d %H:%M:%S}\n\n')

    f.write('DATASET STATISTICS\n' + '-'*80 + '\n')
    f.write(f'Total sentences annotated   : {total:,}\n')
    f.write(f'Sentences with triplets     : {with_t:,}\n')
    f.write(f'Sentences without triplets  : {total-with_t:,}\n')
    f.write(f'Coverage                    : {100*with_t/__builtins__.max(total,1):.1f}%\n\n')

    f.write('TRIPLET STATISTICS\n' + '-'*80 + '\n')
    f.write(f'Total triplets extracted    : {n_t:,}\n')
    f.write(f'Avg triplets/sentence (w/t) : {avg_t:.2f}\n')
    f.write(f'Sentences with >1 triplet   : {multi_t:,}\n')
    f.write(f'Max triplets per sentence   : {max_t}\n\n')

    f.write('SENTIMENT DISTRIBUTION\n' + '-'*80 + '\n')
    for k, name in [(0,'Negative'),(1,'Neutral'),(2,'Positive')]:
        pct = 100*sent_dist[k]/__builtins__.max(total_t,1)
        f.write(f'{name:10} ({k}): {sent_dist[k]:8,}  ({pct:5.1f}%)\n')



Sentences annotated        : 800
Sentences với triplet      : 495
Sentences không có triplet : 305
Coverage                   : 61.9%
Total triplets             : 981
Avg triplets/sentence(w/t) : 1.98
Sentences có >1 triplet    : 301
Max triplets/sentence      : 9
Sentiment – Neg/Neu/Pos    : 345 / 66 / 570


In [ ]:
report_path = str(Path(CFG['report_dir']) / f'hard_labeling_report_{CFG["category_name"]}.txt')
with open(report_path, 'w', encoding='utf-8') as f:

    f.write(f'Total sentences annotated   : {total:,}\n')
    f.write(f'Sentences with triplets     : {with_t:,}\n')
    f.write(f'Sentences without triplets  : {total-with_t:,}\n')
    f.write(f'Coverage                    : {100*with_t/__builtins__.max(total,1):.1f}%\n\n')

    f.write('TRIPLET STATISTICS\n')
    f.write(f'Total triplets extracted    : {n_t:,}\n')
    f.write(f'Avg triplets/sentence (w/t) : {avg_t:.2f}\n')
    f.write(f'Sentences with >1 triplet   : {multi_t:,}\n')
    f.write(f'Max triplets per sentence   : {max_t}\n\n')

    f.write('SENTIMENT DISTRIBUTION\n')
    for k, name in [(0,'Negative'),(1,'Neutral'),(2,'Positive')]:
        pct = 100*sent_dist[k]/__builtins__.max(total_t,1)
        f.write(f'{name:10} ({k}): {sent_dist[k]:8,}  ({pct:5.1f}%)\n')